# Post-processing: Filter Senses with 'FIRST' in SENSE_ORIGIN

This notebook loads all output TSV files, filters sentences where the `SENSE_ORIGIN` field contains 'FIRST' (i.e., the LLM did not find a sense and assigned the first sense from the lexicon), and saves the filtered sentences to separate output files for each input. This prepares the data for a second round of processing or annotation.

## 1. Import Required Libraries
Import necessary libraries for file handling and data processing.

In [1]:
import os
import pandas as pd
from pathlib import Path
import sys
sys.path.append(str(Path().resolve()))

# Import config values if available
try:
    import config
    OUTPUT_DIR = config.OUTPUT_DIR
    SENSE_ORIGIN = config.SENSE_ORIGIN
    SENSE_AINOTES_FIELD = config.SENSE_AINOTES_FIELD
except ImportError:
    OUTPUT_DIR = Path('output')
    SENSE_ORIGIN = 'Origine'  # fallback

# Import the correct WebAnno parsing logic and models
from webanno_spacy_converter.models.sentence_with_mwes import (
    AnnotatedSentenceWithMWEs,
    MultiWordExpression,
)
from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
from writers import CustomWebAnnoTSVWriter, IncetprionWebAnnoTSVWriter

## 2. Process and Filter Output Files
Each output TSV file is loaded, filtered for sentences where the `SENSE_ORIGIN` field contains 'FIRST', and the filtered results are saved to separate output files. This is done for each input file individually, rather than loading all files into a single DataFrame.

In [2]:
from typing import List

def create_file_name(begin: int, end: int, origin_llm: str) -> str:
    """
    Create a file name based on the given parameters, zero-padded to 4 digits.
    Args:
        begin (int): The starting index for the file name.
        end (int): The ending index for the file name.
        origin_llm (str): The origin of the LLM.
    Returns:
        str: A formatted file name.
    """
    return f"LexiSense_Inception_test_{str(begin).zfill(4)}_{str(end).zfill(4)}_{origin_llm}.tsv"

def filter_sentences_with_first_origin(sentences: List[AnnotatedSentenceWithMWEs], sense_origin_field: str) -> List[AnnotatedSentenceWithMWEs]:
    """
    Filter sentences where at least one token has 'FIRST' in the SENSE_ORIGIN field.
    """
    filtered = []
    for sentence in sentences:
        for token in sentence.tokens:
            token_sense_origin = token.layers.get(sense_origin_field, None)
            if token_sense_origin and "FIRST" in token_sense_origin:
                filtered.append(sentence)
                break
    return filtered

def process_files(output_dir: Path, origin_llm: str, begins: List[int], ends: List[int], sense_origin_field: str):
    """
    Process each input file, filter sentences, and save to separate output files.
    """
    for begin, end in zip(begins, ends):
        file_name = create_file_name(begin, end, origin_llm)
        file_path = output_dir / file_name

        if not file_path.exists():
            print(f"File {file_path} does not exist. Skipping...")
            continue
        print(f"Processing file: {file_path}")
        parser = WebAnnoLEXISParser(file_path)
        sentences = parser.parse()

        filtered_sentences = filter_sentences_with_first_origin(sentences, sense_origin_field)
        print(f"Total sentences with 'FIRST' sense origin in {file_name}: {len(filtered_sentences)}")

        filtered_output_path = output_dir / f"filtered_first_origin_{origin_llm}_{str(begin).zfill(4)}_{str(end).zfill(4)}.tsv"
        filtered_incept_path = output_dir / f"filtered_first_origin_incept_{origin_llm}_{str(begin).zfill(4)}_{str(end).zfill(4)}.tsv"

        # Save in standard format
        custom_writer = CustomWebAnnoTSVWriter(filtered_sentences)
        custom_writer.save(filtered_output_path)
        print(f"Filtered sentences saved to {filtered_output_path}")

        # Save in Inception format
        incept_writer = IncetprionWebAnnoTSVWriter(filtered_sentences)
        incept_writer.save(filtered_incept_path)
        print(f"Filtered sentences (Inception) saved to {filtered_incept_path}")

# Call main logic directly for notebook usage
origin_llm = "ChatGPT-4-1"
begins = [1, 501, 1001, 1501, 2001]
ends = [500, 1000, 1500, 2000, 2025]
process_files(OUTPUT_DIR, origin_llm, begins, ends, SENSE_ORIGIN)







Processing file: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_0001_0500_ChatGPT-4-1.tsv
Total sentences with 'FIRST' sense origin in LexiSense_Inception_test_0001_0500_ChatGPT-4-1.tsv: 156
Filtered sentences saved to e:\Github\LexiSense-SR\output\filtered_first_origin_ChatGPT-4-1_0001_0500.tsv
Filtered sentences (Inception) saved to e:\Github\LexiSense-SR\output\filtered_first_origin_incept_ChatGPT-4-1_0001_0500.tsv
Processing file: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_0501_1000_ChatGPT-4-1.tsv
Total sentences with 'FIRST' sense origin in LexiSense_Inception_test_0501_1000_ChatGPT-4-1.tsv: 143
Filtered sentences saved to e:\Github\LexiSense-SR\output\filtered_first_origin_ChatGPT-4-1_0501_1000.tsv
Filtered sentences (Inception) saved to e:\Github\LexiSense-SR\output\filtered_first_origin_incept_ChatGPT-4-1_0501_1000.tsv
Processing file: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_1001_1500_ChatGPT-4-1.tsv
Total sentences with 'FIRST' sense origin 